In [ ]:
# Music For Robots (M4R) — Experiment N (M4R-14):
# Inside the Listening Mind — Mechanistic Token Entropy & N=20 Statistically Powered Proof
#
# Why this experiment is built to publication standard:
#   1. N = 20 independent sampling seeds per condition (80 independent runs total across 4 RMS-matched arms)
#   2. Direct mechanistic measurement: exact per-token Shannon entropy H(p) from raw model logits
#   3. Real creative + structural task (not a toy "uses for a brick" list):
#      "Invent an original 3-sentence concept for a tool or instrument that does not exist yet."
#   4. Full non-parametric statistics: Mann-Whitney U p-values, Cohen's d effect sizes, 95% bootstrap CIs
#   5. Publication-ready figure (m4r14_figure.png) showing exact distributions & 95% CIs

import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers", "bitsandbytes", "accelerate", "librosa", "soundfile",
    "scikit-learn", "sentence-transformers", "scipy", "matplotlib"
])

import json, re
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn.functional as F
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration, BitsAndBytesConfig

DATA_ROOT = Path("/tmp/m4r-data")
if not (DATA_ROOT / "tracks" / "stuttering-packets").exists():
    if DATA_ROOT.exists():
        subprocess.check_call(["rm", "-rf", str(DATA_ROOT)])
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/luchtan-GIT/m4r-data.git", str(DATA_ROOT)])

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
SR = processor.feature_extractor.sampling_rate

y_m4r, _ = librosa.load(str(DATA_ROOT / "tracks" / "stuttering-packets" / "audio.mp3"), sr=SR, mono=True)
target_rms = float(np.sqrt(np.mean(y_m4r ** 2))) + 1e-8

def make_click_track(n_samples: int, bpm: float = 120.0, sr: int = SR):
    y = np.zeros(n_samples, dtype=np.float32)
    step = int((60.0 / bpm) * sr)
    clen = int(0.015 * sr)
    t = np.arange(clen) / sr
    ping = (0.35 * np.sin(2 * np.pi * 880 * t) * np.exp(-t * 140)).astype(np.float32)
    for pos in range(0, n_samples - clen, step):
        y[pos:pos + clen] += ping
    rms = float(np.sqrt(np.mean(y ** 2))) + 1e-8
    return np.clip(y * (target_rms / rms), -0.95, 0.95).astype(np.float32)

def make_white_noise(n_samples: int, seed: int = 42):
    rng = np.random.default_rng(seed)
    y = rng.standard_normal(n_samples).astype(np.float32)
    rms = float(np.sqrt(np.mean(y ** 2))) + 1e-8
    return np.clip(y * (target_rms / rms), -0.95, 0.95).astype(np.float32)

CONDITIONS = {
    "C0_ColdBoot": None,
    "C1_Click_120BPM": make_click_track(len(y_m4r)),
    "C2_StutteringPackets": y_m4r.astype(np.float32),
    "C3_WhiteNoise": make_white_noise(len(y_m4r)),
}

PROMPT = (
    "Invent an original 3-sentence concept for a creative tool or instrument that does not exist yet. "
    "Be concrete about what it does and how a person interacts with it."
)
SEEDS = list(range(101, 121))  # N = 20 independent seeds per condition (80 runs total!)

def generate_with_logit_entropy(conversation, audios, seed=101, max_new_tokens=140):
    torch.manual_seed(seed)
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    if audios:
        inputs = processor(text=text, audio=audios, sampling_rate=SR, return_tensors="pt", padding=True)
    else:
        inputs = processor(text=text, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.92,
            output_scores=True,
            return_dict_in_generate=True,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    gen_ids = out.sequences[:, inputs["input_ids"].shape[1]:]
    reply = processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()

    # Exact per-token Shannon entropy (in bits) from raw pre-softmax logits
    entropies = []
    for step_logits in out.scores:
        logits_f32 = step_logits[0].float()
        probs = F.softmax(logits_f32, dim=-1)
        log_probs = F.log_softmax(logits_f32, dim=-1)
        h_nats = -(probs * log_probs).sum().item()
        entropies.append(h_nats / np.log(2.0))  # bits
    mean_h = float(np.mean(entropies)) if entropies else 0.0
    p90_h = float(np.percentile(entropies, 90)) if entropies else 0.0
    first10_h = float(np.mean(entropies[:10])) if entropies else 0.0
    return reply, mean_h, p90_h, first10_h, len(entropies)

rows = []
for cond_name, y_audio in CONDITIONS.items():
    print(f"\n=== CONDITION: {cond_name} (N={len(SEEDS)} seeds) ===")
    for s in SEEDS:
        conv = []
        audios = []
        if y_audio is not None:
            audios = [y_audio]
            conv.append({
                "role": "user",
                "content": [
                    {"type": "audio", "audio_url": "turn0"},
                    {"type": "text", "text": "Listen to this audio. Simply take it in and say 'Ready.' when it finishes."},
                ],
            })
            # Genuine Turn-0 wake turn!
            wake_rep, _, _, _, _ = generate_with_logit_entropy(conv, audios, seed=s, max_new_tokens=20)
            conv.append({"role": "assistant", "content": wake_rep})
        conv.append({"role": "user", "content": PROMPT})
        rep, mean_h, p90_h, f10_h, n_tok = generate_with_logit_entropy(conv, audios, seed=s + 500, max_new_tokens=140)
        words = re.findall(r"[a-z']+", rep.lower())
        sents = [x.strip() for x in re.split(r"[.!?]+", rep) if len(x.strip().split()) >= 2]
        slens = [len(x.split()) for x in sents] if sents else [len(words)]
        burst = float(np.std(slens) / (np.mean(slens) + 1e-9)) if len(slens) > 1 else 0.0
        ttr = len(set(words)) / max(1, len(words))
        rows.append({
            "condition": cond_name,
            "seed": s,
            "mean_entropy_bits": round(mean_h, 4),
            "first10_entropy_bits": round(f10_h, 4),
            "p90_entropy_bits": round(p90_h, 4),
            "n_tokens": n_tok,
            "burstiness": round(burst, 4),
            "lexical_ttr": round(ttr, 4),
            "response": rep.replace("\n", " "),
        })
        if s in (101, 110, 120):
            print(f"  [s={s}] H={mean_h:.3f}b burst={burst:.3f} -> {rep[:85]!r}")

del model
torch.cuda.empty_cache()

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
df = pd.DataFrame(rows)
embs = embedder.encode(df["response"].tolist(), normalize_embeddings=True)

# Per-run semantic novelty = mean cosine distance to the other 19 runs in the SAME condition (20 values/condition!)
novelties = []
for idx, r in df.iterrows():
    peers = df[(df["condition"] == r["condition"]) & (df.index != idx)].index.tolist()
    sims = embs[peers] @ embs[idx]
    novelties.append(round(float(1.0 - np.mean(sims)), 4))
df["semantic_novelty"] = novelties

# Distance from C0 centroid (how far each concept escapes Cold Boot's average attractor!)
c0_centroid = np.mean(embs[df["condition"] == "C0_ColdBoot"], axis=0)
c0_centroid = c0_centroid / (np.linalg.norm(c0_centroid) + 1e-9)
df["escape_from_c0"] = [round(float(1.0 - np.dot(e, c0_centroid)), 4) for e in embs]

df.to_csv("/kaggle/working/experiment_n_results.csv", index=False)

def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled = np.sqrt(((na - 1)*np.var(a, ddof=1) + (nb - 1)*np.var(b, ddof=1)) / (na + nb - 2))
    return float((np.mean(a) - np.mean(b)) / (pooled + 1e-9))

def boot_ci(vals, n_boot=2000):
    rng = np.random.default_rng(0)
    means = [np.mean(rng.choice(vals, size=len(vals), replace=True)) for _ in range(n_boot)]
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

lines = ["# Experiment N (M4R-14) — Inside the Listening Mind (N=20 Seeds/Arm, 80 Runs, Raw Logit Entropy + p-Values)\n"]
lines.append("## Summary Table (Mean [95% Bootstrap CI])\n")
lines.append("| Condition (N=20) | Logit Entropy (bits) | Semantic Novelty (Within-Arm) | Escape from C0 Centroid | Burstiness (CV) | Lexical TTR |")
lines.append("|---|---:|---:|---:|---:|---:|")
for cond in CONDITIONS:
    g = df[df["condition"] == cond]
    h_lo, h_hi = boot_ci(g["mean_entropy_bits"].values)
    n_lo, n_hi = boot_ci(g["semantic_novelty"].values)
    e_lo, e_hi = boot_ci(g["escape_from_c0"].values)
    lines.append(
        f"| `{cond}` | {g['mean_entropy_bits'].mean():.3f} [{h_lo:.3f}, {h_hi:.3f}] | "
        f"{g['semantic_novelty'].mean():.4f} [{n_lo:.4f}, {n_hi:.4f}] | "
        f"{g['escape_from_c0'].mean():.4f} [{e_lo:.4f}, {e_hi:.4f}] | "
        f"{g['burstiness'].mean():.4f} | {g['lexical_ttr'].mean():.4f} |"
    )

lines.append("\n## Statistical Significance Tests (Mann-Whitney U Two-Sided + Cohen's d)\n")
lines.append("| Comparison | Metric | U-stat | p-value | Cohen's d |")
lines.append("|---|---|---:|---:|---:|")
m4r = df[df["condition"] == "C2_StutteringPackets"]
for other_name in ["C0_ColdBoot", "C1_Click_120BPM", "C3_WhiteNoise"]:
    oth = df[df["condition"] == other_name]
    for col in ["semantic_novelty", "mean_entropy_bits", "escape_from_c0", "burstiness"]:
        u, p = stats.mannwhitneyu(m4r[col], oth[col], alternative="two-sided")
        d = cohens_d(m4r[col].values, oth[col].values)
        lines.append(f"| `StutteringPackets` vs `{other_name}` | `{col}` | {u:.1f} | {p:.6f} | {d:+.3f} |")

# Also save 3 sample inventions per condition
lines.append("\n## Sample Inventions (Seed 101)\n")
for cond in CONDITIONS:
    r = df[(df["condition"] == cond) & (df["seed"] == 101)].iloc[0]
    lines.append(f"- **`{cond}`** (H={r['mean_entropy_bits']:.2f} bits, novelty={r['semantic_novelty']:.3f}): {r['response']}")

Path("/kaggle/working/experiment_n_summary.md").write_text("\n".join(lines), encoding="utf-8")

# Publication-grade dark Obsidian + Orange figure
plt.style.use("dark_background")
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), facecolor="#111113")
cond_order = ["C0_ColdBoot", "C1_Click_120BPM", "C2_StutteringPackets", "C3_WhiteNoise"]
labels = ["Silence\n(Cold Boot)", "120 BPM Click\n(Rigid Order)", "Stuttering Packets\n(Live Unquantized)", "White Noise\n(Pure Chaos)"]
colors = ["#66666e", "#8e8e98", "#ff7216", "#8e8e98"]

for ax, col, title, ylabel in [
    (axes[0], "semantic_novelty", "Concept Originality Across N=20 Seeds", "Within-Arm Semantic Novelty (95% CI)"),
    (axes[1], "mean_entropy_bits", "Raw Next-Token Logit Entropy (N=20)", "Mean Shannon Entropy (bits, 95% CI)"),
]:
    ax.set_facecolor("#161619")
    means, err_lo, err_hi = [], [], []
    for c in cond_order:
        vals = df[df["condition"] == c][col].values
        m = float(np.mean(vals))
        lo, hi = boot_ci(vals)
        means.append(m); err_lo.append(m - lo); err_hi.append(hi - m)
    x = np.arange(len(cond_order))
    bars = ax.bar(x, means, yerr=[err_lo, err_hi], capsize=6, color=colors, edgecolor="#f4f4f2", linewidth=0.7)
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9.5, color="#f4f4f2")
    ax.set_title(title, fontsize=11.5, fontweight="bold", color="#f4f4f2", pad=12)
    ax.set_ylabel(ylabel, fontsize=10, color="#c8c8d0")
    ax.grid(axis="y", alpha=0.18, linestyle="--")
    for b, m in zip(bars, means):
        ax.text(b.get_x() + b.get_width()/2, m + max(err_hi)*1.15, f"{m:.3f}", ha="center", va="bottom", fontsize=9.5, fontweight="bold", color="#f4f4f2")

fig.suptitle("M4R-14: Inside the Listening Mind (Qwen2-Audio-7B · N=20 Seeds/Arm · RMS-Matched 200s)", fontsize=13, fontweight="bold", color="#ff7216", y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig("/kaggle/working/m4r14_figure.png", dpi=180, facecolor="#111113")
print("\nDONE! Total M4R-14 runs:", len(df))
print("\n".join(lines[:32]))
